<a href="https://colab.research.google.com/github/nthgiang311/computer-vision-project/blob/main/Road_Scene_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Environment and imports

In [ ]:
!pip install -q torchmetrics
!pip install -q segmentation-models-pytorch
!pip install -q opencv-python

# Import thư viện
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import files, drive
import random
from collections import Counter
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassJaccardIndex

drive.mount('/content/drive')

2. Mount Drive and prepare dataset

Dataset được clone vào Drive ở lần chạy đầu tiên; những lần sau sẽ dùng lại thư mục có sẵn.

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/road_sence_segmentation'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/output", exist_ok=True)

CAMVID_DIR = f"{PROJECT_DIR}/data/CamVid"

# Chỉ clone nếu chưa có dataset
if not os.path.exists(CAMVID_DIR):
    !git clone -q https://github.com/lih627/CamVid.git "$CAMVID_DIR"
    print("Đã tải dataset vào:", CAMVID_DIR)
else:
    print("Dataset đã tồn tại tại:", CAMVID_DIR)

for root, dirs, files in os.walk(CAMVID_DIR):
    level = root.replace(CAMVID_DIR, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    # Chỉ duyệt sâu tối đa 2 cấp
    if level >= 2:
        dirs[:] = []

3. Inspect dataset and label mapping


In [ ]:
image_path = Path(f"{CAMVID_DIR}/CamVid_RGB/0001TP_007950.png")
mask_path = Path(f"{CAMVID_DIR}/CamVidColor11/0001TP_007950_L.png")

image = Image.open(image_path)
mask = Image.open(mask_path)

print("Image size:", image.size)
print("Mask size:", mask.size)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask)
plt.title("Segmentation Mask")
plt.axis("off")

plt.show()


image_dir = Path(f"{CAMVID_DIR}/CamVid_RGB")
mask_dir = Path(f"{CAMVID_DIR}/CamVidColor11")

image_files = sorted(image_dir.glob("*.png"))
mask_files = sorted(mask_dir.glob("*.png"))

print("Number of images:", len(image_files))
print("Number of masks:", len(mask_files))

image_names = {p.stem for p in image_files}
mask_names = {p.stem.replace("_L", "") for p in mask_files}

missing_masks = image_names - mask_names
missing_images = mask_names - image_names

print("Images without masks:", len(missing_masks))
print("Masks without images:", len(missing_images))

if missing_masks:
    print("Missing masks:", sorted(missing_masks)[:10])

if missing_images:
    print("Missing images:", sorted(missing_images)[:10])

image_sizes = Counter()
mask_sizes = Counter()

for image_path, mask_path in zip(image_files, mask_files):
    image_sizes[Image.open(image_path).size] += 1
    mask_sizes[Image.open(mask_path).size] += 1

print("Image sizes:")
for size, count in image_sizes.items():
    print(size, ":", count)

print("\nMask sizes:")
for size, count in mask_sizes.items():
    print(size, ":", count)

valid_ids = set(range(11)) | {255}

total_pixel_counts = Counter()
invalid_ids = set()

for mask_path in mask_files:
    mask = np.array(Image.open(mask_path))

    unique_ids, counts = np.unique(mask, return_counts=True)

    for class_id, count in zip(unique_ids, counts):
        total_pixel_counts[int(class_id)] += int(count)

    invalid_ids.update(set(unique_ids) - valid_ids)

print("Invalid label IDs:", invalid_ids)

print("\nPixel count of each label:")
for class_id in sorted(total_pixel_counts):
    print(f"ID {class_id}: {total_pixel_counts[class_id]:,}")

sample_mask = Image.open(mask_files[0])

print("Mode:", sample_mask.mode)
print("Size:", sample_mask.size)

mask_array = np.array(sample_mask)

print("Unique values:")
print(np.unique(mask_array))

class_names = [
    "Sky",          # 0
    "Building",     # 1
    "Pole",         # 2
    "Road",         # 3
    "Sidewalk",     # 4
    "Tree",         # 5
    "SignSymbol",   # 6
    "Fence",        # 7
    "Car",          # 8
    "Pedestrian",   # 9
    "Bicyclist"     # 10
]

num_classes = len(class_names)

print("Number of classes:", num_classes)

for i, name in enumerate(class_names):
    print(i, "->", name)

# Chọn một sample
idx = 0

image = np.array(Image.open(image_files[idx]).convert("RGB"))
mask = np.array(Image.open(mask_files[idx]))

print("Image shape:", image.shape)
print("Mask shape :", mask.shape)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask)
plt.title("Mask (Class IDs)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(image)
plt.imshow(mask, alpha=0.5)
plt.title("Image + Mask")
plt.axis("off")

plt.show()

CAMVID_COLORS = [
    (128, 128, 128),  # Sky
    (128, 0, 0),      # Building
    (192, 192, 128),  # Pole
    (128, 64, 128),   # Road
    (0, 0, 192),      # Sidewalk
    (128, 128, 0),    # Tree
    (192, 128, 128),  # SignSymbol
    (64, 64, 128),    # Fence
    (64, 0, 128),     # Car
    (64, 64, 0),      # Pedestrian
    (0, 128, 192),    # Bicyclist
]

color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)

for class_id, color in enumerate(CAMVID_COLORS):
    color_mask[mask == class_id] = color

# Void / Ignore
color_mask[mask == 255] = (0, 0, 0)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(color_mask)
plt.title("Color Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(image)
plt.imshow(color_mask, alpha=0.5)
plt.title("Image + Color Mask")
plt.axis("off")

plt.show()

4. Load train / validation / test splits

Đọc các file split của CamVid và kiểm tra ảnh-mask có khớp nhau hay không.

In [ ]:
split_dir = Path(CAMVID_DIR)

for split_name in ["train", "val", "test"]:
    split_file = split_dir / f"camvid_{split_name}.txt"

    print(f"\n===== {split_name.upper()} =====")
    with open(split_file, "r") as f:
        lines = f.readlines()

    print("Number of samples:", len(lines))
    print("First 5 lines:")
    for line in lines[:5]:
        print(line.strip())

from pathlib import Path

split_dir = Path(CAMVID_DIR)

def load_split(split_name):
    split_file = split_dir / f"camvid_{split_name}.txt"

    image_paths = []

    with open(split_file, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # Mỗi dòng có:
            # CamVid_RGB/image.png CamVidGray/mask.png
            image_rel_path = line.split()[0]

            image_name = Path(image_rel_path).name

            image_path = image_dir / image_name
            mask_path = mask_dir / image_name.replace(".png", "_L.png")

            image_paths.append((image_path, mask_path))

    return image_paths


train_pairs = load_split("train")
val_pairs = load_split("val")
test_pairs = load_split("test")

print("Train:", len(train_pairs))
print("Val  :", len(val_pairs))
print("Test :", len(test_pairs))

def check_pairs(pairs, split_name):
    missing_images = []
    missing_masks = []

    for image_path, mask_path in pairs:
        if not image_path.exists():
            missing_images.append(image_path.name)

        if not mask_path.exists():
            missing_masks.append(mask_path.name)

    print(f"\n===== {split_name.upper()} =====")
    print("Missing images:", len(missing_images))
    print("Missing masks :", len(missing_masks))

    if missing_images:
        print("Example missing images:", missing_images[:5])

    if missing_masks:
        print("Example missing masks:", missing_masks[:5])


check_pairs(train_pairs, "train")
check_pairs(val_pairs, "val")
check_pairs(test_pairs, "test")

train_names = {image_path.name for image_path, _ in train_pairs}
val_names   = {image_path.name for image_path, _ in val_pairs}
test_names  = {image_path.name for image_path, _ in test_pairs}

print("Train ∩ Val :", len(train_names & val_names))
print("Train ∩ Test:", len(train_names & test_names))
print("Val ∩ Test  :", len(val_names & test_names))

5. Preprocessing and augmentation

Ảnh được resize bằng bilinear; mask dùng nearest-neighbor để không tạo ra class ID mới. Các phép biến đổi hình học được áp dụng đồng thời lên ảnh và mask.

In [ ]:
class SegmentationTransform:
    def __init__(self, img_size=(256, 192), augment=False):
        self.img_size = img_size
        self.augment = augment

        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __call__(self, image, mask):

        # 1. RESIZE
        image = TF.resize(
            image,
            self.img_size,
            interpolation=InterpolationMode.BILINEAR
        )

        mask = TF.resize(
            mask,
            self.img_size,
            interpolation=InterpolationMode.NEAREST
        )

        # 2. AUGMENTATION
        if self.augment:

            # Horizontal Flip
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            # Rotation
            if random.random() < 0.5:
                angle = random.uniform(-15, 15)

                image = TF.rotate(
                    image,
                    angle,
                    interpolation=InterpolationMode.BILINEAR,
                    fill = 0
                )

                mask = TF.rotate(
                    mask,
                    angle,
                    interpolation=InterpolationMode.NEAREST,
                    fill = 255
                )

            # Brightness
            if random.random() < 0.5:
                brightness = random.uniform(0.8, 1.2)
                image = TF.adjust_brightness(
                    image,
                    brightness
                )

            # Contrast
            if random.random() < 0.5:
                contrast = random.uniform(0.8, 1.2)
                image = TF.adjust_contrast(
                    image,
                    contrast
                )

        # 3. IMAGE -> TENSOR
       image = TF.to_tensor(image)

        image = TF.normalize(
            image,
            mean=self.mean,
            std=self.std
        )

        # 4. MASK -> TENSOR
        mask = torch.from_numpy(
            np.array(mask)
        ).long()

        return image, mask

train_transform = SegmentationTransform(
    img_size=(384, 512),
    augment=True
)

val_transform = SegmentationTransform(
    img_size=(384, 512),
    augment=False
)

test_transform = SegmentationTransform(
    img_size=(384, 512),
    augment=False
)

image_path, mask_path = train_pairs[0]

image = Image.open(image_path).convert("RGB")
mask = Image.open(mask_path)

image_tensor, mask_tensor = train_transform(image, mask)

print("Image:")
print("  shape:", image_tensor.shape)
print("  dtype:", image_tensor.dtype)
print("  min:", image_tensor.min().item())
print("  max:", image_tensor.max().item())

print("\nMask:")
print("  shape:", mask_tensor.shape)
print("  dtype:", mask_tensor.dtype)
print("  unique values:", torch.unique(mask_tensor))

# Unnormalize image để hiển thị đúng màu
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

image_display = image_tensor * std + mean
image_display = image_display.clamp(0, 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_display.permute(1, 2, 0))
plt.title("Transformed Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask_tensor)
plt.title("Transformed Mask")
plt.axis("off")

plt.show()

6. Dataset and DataLoaders

Tạo Dataset cho ba tập dữ liệu và kiểm tra shape trước khi huấn luyện.

In [ ]:
class CamVidDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        # Đọc ảnh
        image = Image.open(image_path).convert("RGB")

        # Đọc mask, giữ nguyên label ID
        mask = Image.open(mask_path)

        if self.transform is not None:
            image, mask = self.transform(image, mask)

        return image, mask

train_dataset = CamVidDataset(
    train_pairs,
    transform=train_transform
)

val_dataset = CamVidDataset(
    val_pairs,
    transform=val_transform
)

test_dataset = CamVidDataset(
    test_pairs,
    transform=test_transform
)

image, mask = train_dataset[0]

print("Image:")
print("  shape:", image.shape)
print("  dtype:", image.dtype)

print("\nMask:")
print("  shape:", mask.shape)
print("  dtype:", mask.dtype)
print("  unique:", torch.unique(mask))

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, masks = next(iter(train_loader))

print("Images:")
print("  shape:", images.shape)
print("  dtype:", images.dtype)

print("\nMasks:")
print("  shape:", masks.shape)
print("  dtype:", masks.dtype)

print("\nMask unique values:")
print(torch.unique(masks))


7. Training and validation utilities

Các hàm cho một epoch huấn luyện, validation, lưu lịch sử loss và vẽ learning curves.

In [ ]:
def train_one_epoch(model,loader,criterion,optimizer,device):
    model.train()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        # Xóa gradient của batch trước
        optimizer.zero_grad()

        # Forward
        outputs = model(images)

        # Tính loss
        loss = criterion(outputs, masks)

        # Backward
        loss.backward()

        # Cập nhật trọng số
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    return average_loss

def validate(
    model,
    loader,
    criterion,
    device,
    miou_metric
):
    model.eval()

    total_loss = 0.0
    correct_pixels = 0
    total_valid_pixels = 0

    # Reset mIoU trước mỗi lần validation
    miou_metric.reset()

    with torch.no_grad():
        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device).long()

            # Forward
            outputs = model(images)

            # Loss
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            # Prediction: [B, H, W]
            preds = torch.argmax(outputs, dim=1)

            # Tính mIoU
            miou_metric.update(preds, masks)

            # Tính Pixel Accuracy, bỏ qua pixel có nhãn 255
            valid = masks != 255

            correct_pixels += (
                (preds == masks) & valid
            ).sum().item()

            total_valid_pixels += valid.sum().item()

    # Trung bình loss trên các batch
    average_loss = (
        total_loss / len(loader)
        if len(loader) > 0 else float("nan")
    )

    # mIoU trên toàn bộ validation set
    miou = miou_metric.compute().item()

    # Pixel Accuracy trên toàn bộ validation set
    pixel_accuracy = (
        correct_pixels / total_valid_pixels
        if total_valid_pixels > 0 else float("nan")
    )

    return average_loss, miou, pixel_accuracy

history_path = f"{PROJECT_DIR}/output/loss_history.csv"

def save_loss(path, epoch, train_loss, val_loss):
    columns = ["epoch", "train_loss", "val_loss"]

    if os.path.exists(path):
        history = pd.read_csv(path)
    else:
        history = pd.DataFrame(columns=columns)

    # Tránh lưu trùng epoch khi chạy lại
    history = history[history["epoch"] != epoch]

    new_row = pd.DataFrame([{
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    }])

    history = pd.concat(
        [history, new_row],
        ignore_index=True
    ).sort_values("epoch")

    history.to_csv(path, index=False)

    return history

def plot_loss(path=history_path):
    if not os.path.exists(path):
        print("Chưa có file lịch sử loss.")
        return

    history = pd.read_csv(path).sort_values("epoch")

    if history.empty:
        print("Lịch sử loss đang trống.")
        return

    plt.figure(figsize=(10, 5))

    plt.plot(
        history["epoch"],
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        history["epoch"],
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


8. Model setup and training

Mô hình U-Net dùng encoder ResNet34 pretrained ImageNet. Loss kết hợp Cross Entropy và Dice Loss; checkpoint tốt nhất được chọn theo validation mIoU.
resume = True sẽ tiếp tục từ checkpoint nếu file tồn tại. Đặt thành False nếu muốn huấn luyện lại từ đầu.

In [ ]:
num_classes = 11
num_epoches = 10
resume = True
checkpoint_path = f"{PROJECT_DIR}/output/checkpoint.pth"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Loss
ce_loss = nn.CrossEntropyLoss(
    ignore_index=255
)
dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    ignore_index=255
)
def criterion(pred, target):
    return (
        ce_loss(pred, target)
        + dice_loss(pred, target)
    )

#mIoU
miou_metric = MulticlassJaccardIndex(
    num_classes,
    ignore_index=255
).to(device)

# Model
#model = UNet(num_classes).to(device)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=11
).to(device)

# Freeze toàn bộ model
#for param in model.parameters():
#    param.requires_grad = False

# Optimizer
optimizer = torch.optim.Adam([
    {
        "params": model.encoder.parameters(),
        "lr": 1e-4
    },
    {
        "params": model.decoder.parameters(),
        "lr": 1e-3
    },
    {
        "params": model.segmentation_head.parameters(),
        "lr": 1e-3
    }
], weight_decay=1e-4)

# Scheduler Learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Load checkpoint
if resume and os.path.exists(checkpoint_path):

    checkpoint = torch.load(
    f"{PROJECT_DIR}/output/checkpoint.pth",
    map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"] + 1

    best_val_miou = checkpoint["best_val_miou"]

else:
    start_epoch = 0
    best_val_miou = -1
    print("Train from scratch")

train_losses = []
val_losses = []
val_mious = []
val_pixel_accs = []
early_stopping = 0

for epoch in range(start_epoch, num_epoches):

    # TRAIN
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    # VALIDATION
    val_loss, val_miou, val_pixel_acc = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        miou_metric=miou_metric
    )

    # SAVE HISTORY
    save_loss(
    path=history_path,
    epoch=epoch + 1,
    train_loss=train_loss,
    val_loss=val_loss
)
    val_mious.append(val_miou)
    val_pixel_accs.append(val_pixel_acc)

    # SAVE BEST MODEL
    if val_miou > best_val_miou:

        best_val_miou = val_miou
        early_stopping = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_miou": best_val_miou,
        }, f"{PROJECT_DIR}/output/checkpoint.pth")

        best_mark = " <-- BEST"

    else:

        best_mark = ""
        early_stopping += 1

    # Scheduler gọi mỗi epoch
    scheduler.step(val_miou)

    # Early stopping
    if early_stopping > 10:
        print("Early stopping")
        break

    print(
        f"Epoch [{epoch + 1:02d}/{num_epoches}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val mIoU: {val_miou:.4f} "
        f"| Val Pixel Acc: {val_pixel_acc:.4f}"
        f"{best_mark}"
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_val_miou = checkpoint["best_val_miou"]

model.eval()

print(
    f"Best Validation mIoU: "
    f"{best_val_miou:.4f}"
)
plot_loss()


9. Evaluation and visualization functions

Định nghĩa các hàm đánh giá trên test set, chuyển mask sang màu, hiển thị dự đoán và vẽ confusion matrix. Với confusion matrix, hàng là nhãn thật và cột là nhãn dự đoán.

In [ ]:
def test_model(
    model,
    loader,
    device,
    miou_metric
):
    model.eval()

    # Reset mIoU trước khi chạy test
    miou_metric.reset()

    correct_pixels = 0
    total_valid_pixels = 0

    with torch.no_grad():
        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device).long()

            # Forward
            outputs = model(images)

            # Prediction: [B, H, W]
            preds = torch.argmax(outputs, dim=1)

            # Tính mIoU
            miou_metric.update(preds, masks)

            # Tính Pixel Accuracy, bỏ qua nhãn 255
            valid = masks != 255

            correct_pixels += (
                (preds == masks) & valid
            ).sum().item()

            total_valid_pixels += valid.sum().item()

    # mIoU trên toàn bộ test set
    miou = miou_metric.compute().item()

    # Pixel Accuracy trên toàn bộ test set
    pixel_accuracy = (
        correct_pixels / total_valid_pixels
        if total_valid_pixels > 0 else float("nan")
    )

    return miou, pixel_accuracy

class_colors = np.array(CAMVID_COLORS)

class_names = [
    "Sky",
    "Building",
    "Pole",
    "Road",
    "Sidewalk",
    "Tree",
    "SignSymbol",
    "Fence",
    "Car",
    "Pedestrian",
    "Bicyclist"
]


def mask_to_color(mask):
    """
    Chuyển mask class ID [H, W] thành ảnh RGB [H, W, 3].
    Các pixel có nhãn 255 được tô đen.
    """
    mask = np.asarray(mask)
    h, w = mask.shape

    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id in range(num_classes):
        color_mask[mask == class_id] = class_colors[class_id]

    color_mask[mask == 255] = [0, 0, 0]

    return color_mask



def visualize_segmentation(
    image,
    pred_mask,
    gt_mask=None,
    ignore_index=255
):
    """
    Có GT:
        Original | Ground Truth | Prediction

    Không có GT:
        Original | Prediction

    Chú giải tên lớp nằm bên cạnh ảnh.
    """

    if isinstance(image, Image.Image):
        image = np.array(image)

    if isinstance(pred_mask, Image.Image):
        pred_mask = np.array(pred_mask)

    image = np.asarray(image)
    pred_mask = np.asarray(pred_mask)

    pred_color = mask_to_color(pred_mask)

    has_gt = gt_mask is not None

    if has_gt:
        if isinstance(gt_mask, Image.Image):
            gt_mask = np.array(gt_mask)

        gt_mask = np.asarray(gt_mask)

        assert gt_mask.shape == pred_mask.shape, (
            "Ground Truth và Prediction phải cùng kích thước."
        )

        gt_color = mask_to_color(gt_mask)

        valid = (
            (gt_mask != ignore_index)
            & (gt_mask >= 0)
            & (gt_mask < num_classes)
        )

        pixel_acc = (
            np.mean(gt_mask[valid] == pred_mask[valid])
            if np.any(valid)
            else float("nan")
        )

        fig, axes = plt.subplots(
            1, 3,
            figsize=(17, 6),
            gridspec_kw={"width_ratios": [1.2, 1, 1]}
        )

        axes[0].imshow(image)
        axes[0].set_title("Original Image")

        axes[1].imshow(gt_color)
        axes[1].set_title("Ground Truth")

        axes[2].imshow(pred_color)
        axes[2].set_title(
            f"Prediction | Pixel Accuracy: {pixel_acc:.2%}"
            if np.isfinite(pixel_acc)
            else "Prediction | Pixel Accuracy: N/A"
        )

        for ax in axes:
            ax.axis("off")

    else:
        # Ảnh tự tải lên, không có GT
        pixel_acc = None

        fig, axes = plt.subplots(
            1, 2,
            figsize=(12, 5)
        )

        axes[0].imshow(image)
        axes[0].set_title("Original Image")

        axes[1].imshow(pred_color)
        axes[1].set_title("Prediction (No Ground Truth)")

        for ax in axes:
            ax.axis("off")

    # Chú giải lớp dùng chung
    legend_handles = [
        Patch(
            facecolor=np.asarray(class_colors[i]) / 255.0,
            edgecolor="none",
            label=class_names[i]
        )
        for i in range(num_classes)
    ]

    legend_handles.append(
        Patch(
            facecolor="black",
            edgecolor="none",
            label=f"Ignore ({ignore_index})"
        )
    )

    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(0.87, 0.5),
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

    return pixel_acc

def plot_confusion_matrix(
    gt_masks,
    pred_masks,
    ignore_index=255,
    normalize=False
):
    """
    gt_masks và pred_masks:
        - Một mask [H, W], hoặc
        - Danh sách mask cùng kích thước tương ứng.

    Hàng: Ground Truth
    Cột: Prediction

    normalize=False: hiển thị số pixel.
    normalize=True: chuẩn hóa từng hàng theo nhãn thật.
    """

    def to_mask_list(masks):
        if isinstance(masks, Image.Image):
            return [np.asarray(masks)]

        if isinstance(masks, np.ndarray):
            if masks.ndim == 2:
                return [masks]
            if masks.ndim == 3:
                return [masks[i] for i in range(masks.shape[0])]

        return [
            np.asarray(m)
            for m in masks
        ]

    gt_list = to_mask_list(gt_masks)
    pred_list = to_mask_list(pred_masks)

    assert len(gt_list) == len(pred_list), (
        "Số lượng Ground Truth và Prediction phải bằng nhau."
    )

    # CM[i, j] = số pixel có nhãn thật i, dự đoán j
    cm = np.zeros(
        (num_classes, num_classes),
        dtype=np.int64
    )

    total_valid = 0
    total_correct = 0

    for gt, pred in zip(gt_list, pred_list):
        assert gt.shape == pred.shape, (
            "Mỗi cặp GT và Prediction phải cùng kích thước."
        )

        valid = (
            (gt != ignore_index)
            & (gt >= 0)
            & (gt < num_classes)
        )

        gt_valid = gt[valid].astype(np.int64)
        pred_valid = pred[valid].astype(np.int64)

        # Prediction của mô hình phải nằm trong [0, num_classes - 1]
        pred_in_range = (
            (pred_valid >= 0)
            & (pred_valid < num_classes)
        )

        gt_valid = gt_valid[pred_in_range]
        pred_valid = pred_valid[pred_in_range]

        counts = np.bincount(
            gt_valid * num_classes + pred_valid,
            minlength=num_classes * num_classes
        )

        cm += counts.reshape(num_classes, num_classes)

        total_valid += len(gt_valid)
        total_correct += np.sum(gt_valid == pred_valid)

    pixel_acc = (
        total_correct / total_valid
        if total_valid > 0
        else float("nan")
    )

    # Dữ liệu để hiển thị
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        display_cm = np.divide(
            cm.astype(float),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0
        )
        fmt = ".2f"
        colorbar_label = "Proportion per true class"
    else:
        display_cm = cm
        fmt = ","
        colorbar_label = "Number of pixels"

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    im = ax.imshow(
        display_cm,
        cmap="Blues",
        interpolation="nearest"
    )

    fig.colorbar(
        im,
        ax=ax,
        label=colorbar_label
    )

    ids = np.arange(num_classes)

    ax.set_xticks(ids)
    ax.set_yticks(ids)
    ax.set_xticklabels(
        class_names,
        rotation=45,
        ha="right"
    )
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Ground Truth Label")
    ax.set_title(
        f"Confusion Matrix | Pixel Accuracy = {pixel_acc:.2%}"
        if np.isfinite(pixel_acc)
        else "Confusion Matrix | Pixel Accuracy = N/A"
    )

    # Hiển thị giá trị trong từng ô
    threshold = (
        display_cm.max() / 2
        if display_cm.size and display_cm.max() > 0
        else 0
    )

    for i in range(num_classes):
        for j in range(num_classes):
            value = display_cm[i, j]

            if normalize:
                text_value = format(value, fmt)
            else:
                text_value = format(int(value), fmt)

            ax.text(
                j, i, text_value,
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value > threshold else "black"
            )

    plt.tight_layout()
    plt.show()

    print(f"Pixel-wise Accuracy: {pixel_acc:.4%}")
    print(f"Valid pixels evaluated: {total_valid:,}")

    return cm, pixel_acc


# IoU cho từng class
def evaluate_per_class_iou(model, loader, device, num_classes):

    model.eval()

    intersection = np.zeros(num_classes)
    union = np.zeros(num_classes)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            for cls in range(num_classes):
                valid = masks != 255
                pred_cls = (preds == cls) & valid
                mask_cls = (masks == cls) & valid

                intersection[cls] += (
                    pred_cls & mask_cls
                ).sum().item()

                union[cls] += (
                    pred_cls | mask_cls
                ).sum().item()


    ious = []

    for cls in range(num_classes):

        if union[cls] == 0:
            iou = float("nan")
        else:
            iou = intersection[cls] / union[cls]

        ious.append(iou)

    return ious


10. Test-set metrics

Tính mIoU, pixel accuracy và IoU theo từng lớp trên tập test.

In [ ]:
test_miou, test_pixel_acc = test_model(
    model=model,
    loader=test_loader,
    device=device,
    miou_metric=miou_metric
)

per_class_iou = evaluate_per_class_iou(
    model=model,
    loader=test_loader,
    device=device,
    num_classes=num_classes
)

print(f"Test mIoU:       {test_miou:.4f}")
print(f"Test Pixel Acc:  {test_pixel_acc:.4f}")
results = list(zip(class_names, per_class_iou))

results = sorted(
    results,
    key=lambda x: x[1],
    reverse=True
)

print("\nPer-Class IoU")
print("-" * 30)

for name, iou in results:
    print(f"{name:12s}: {iou:.4f}")


11. Qualitative results and confusion matrix

Hiển thị một số ảnh test cùng ground truth/prediction, sau đó tổng hợp confusion matrix trên toàn bộ test set.

In [ ]:
# 1. Chọn 5 ảnh test và dự đoán cùng lúc
num_samples = min(5, len(test_dataset))
indices = random.sample(range(len(test_dataset)), num_samples)

samples = [test_dataset[idx] for idx in indices]

images = torch.stack([sample[0] for sample in samples]).to(device)

model.eval()
with torch.no_grad():
    outputs = model(images)
    pred_masks = outputs.argmax(dim=1).cpu().numpy()

# 2. Hiển thị 5 ảnh bằng hàm có sẵn
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i, idx in enumerate(indices):
    image = samples[i][0].cpu().numpy().transpose(1, 2, 0)

    # Khôi phục ảnh sau Normalize ImageNet
    image = np.clip(image * std + mean, 0, 1)

    gt_mask = samples[i][1].cpu().numpy()
    pred_mask = pred_masks[i]

    visualize_segmentation(
        image=image,
        pred_mask=pred_mask,
        gt_mask=gt_mask
    )

    print(f"Test sample index: {idx}")

# 3. Confusion Matrix toàn bộ tập test
all_gt_masks = []
all_pred_masks = []

with torch.no_grad():
    for images_batch, masks_batch in test_loader:
        images_batch = images_batch.to(device)

        outputs = model(images_batch)
        predictions = outputs.argmax(dim=1)

        all_gt_masks.extend(masks_batch.cpu().numpy())
        all_pred_masks.extend(predictions.cpu().numpy())

cm, cm_pixel_acc = plot_confusion_matrix(
    gt_masks=all_gt_masks,
    pred_masks=all_pred_masks,
    ignore_index=255,
    normalize=False
)


12. Inference on a custom image

Upload một ảnh để dự đoán segmentation. Cell này dành cho Google Colab; ảnh không có ground truth nên chỉ hiển thị ảnh đầu vào và mask dự đoán.

In [ ]:
# Upload ảnh
uploaded = files.upload()

# Lấy file đầu tiên
file_name = list(uploaded.keys())[0]

# Đọc ảnh
image = Image.open(file_name).convert("RGB")

# Kích thước ảnh gốc
orig_w, orig_h = image.size

# Preprocess giống validation
dummy_mask = Image.fromarray(
    np.zeros((orig_h, orig_w), dtype=np.uint8)
)

image_tensor, _ = test_transform(
    image,
    dummy_mask
)

# Thêm batch dimension
x = image_tensor.unsqueeze(0).to(device)

# Model prediction
model.eval()

with torch.no_grad():
    logits = model(x)

# Đưa output về kích thước ảnh gốc
logits = F.interpolate(
    logits,
    size=(orig_h, orig_w),
    mode="bilinear",
    align_corners=False
)

# Lấy class có xác suất cao nhất
pred_mask = torch.argmax(
    logits,
    dim=1
).squeeze(0).cpu().numpy()

print("Image:", file_name)
print("Original size:", (orig_w, orig_h))
print("Predicted classes:", np.unique(pred_mask))

visualize_segmentation(
    image,
    pred_mask,
    gt_mask=None,
    ignore_index=255
